# German Language Analysis

This notebook is configured to work with German language datasets and text processing. Ensure all preprocessing, tokenization, and model steps use German language settings or resources.

# GT Keyword Extraction and Processing Workflow

In [22]:
# Configuration
dataset_name = "german"
dataset_language = "german"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 85

# Stemming, Compound Analysis, and Stopword Libraries
from nltk.stem.snowball import SnowballStemmer
import nltk
from nltk.corpus import stopwords  # Stopword library
# German compound splitting using compound-split
from compound_split.doc_split import get_best_split

# Initialize stemmer and stopwords
stemmer = SnowballStemmer(dataset_language)
nltk_german_stopwords = set(stopwords.words(dataset_language))
# Use get_best_split for compound splitting
def compoundsplitter(word):
    return get_best_split(word)


In [23]:
import re
def process_keywords(keywords):
    """Apply cleaning, German compound splitting (compound-split), stemming, and stopword filtering to a list of keywords. Returns a processed list of useful words."""
    processed = []
    for word in keywords:
        # Clean word: keep only alphabetic characters (German letters)
        cleaned_word = re.sub(r'[^a-zA-ZäöüßÄÖÜ]', '', word)
        if not cleaned_word:
            continue
        # Compound splitting (German, using compound-split)
        compound_parts = [cleaned_word]
        try:
            split_result = compoundsplitter(cleaned_word)
            if split_result and isinstance(split_result, list):
                # split_result is a list of parts
                compound_parts.extend([part.lower() for part in split_result if part.lower() != cleaned_word.lower()])
        except Exception:
            pass
        # Clean all compound parts
        cleaned_parts = [re.sub(r'[^a-zA-ZäöüßÄÖÜ]', '', part) for part in compound_parts]
        # Remove empty strings after cleaning
        cleaned_parts = [part for part in cleaned_parts if part]
        # Stem all parts
        stemmed_parts = [stemmer.stem(part) for part in cleaned_parts]
        # Filter stopwords
        filtered = [part for part in stemmed_parts if part not in nltk_german_stopwords]
        processed.extend(filtered)
    # Return unique processed keywords (useful words only)
    return processed


In [24]:
from bs4 import BeautifulSoup
def extract_and_process_keywords_from_tag(html_text, tag_name):
    """Extracts keywords from the content inside the specified tag in the given HTML text, processes them using process_keywords, and returns a dictionary with keyword frequencies."""
    soup = BeautifulSoup(html_text, 'html.parser')
    elements = soup.find_all(tag_name)
    raw_keywords = []
    for elem in elements:
        text = elem.get_text(separator=' ')
        raw_keywords.extend(text.split())
    processed = process_keywords(raw_keywords)
    freq = {}
    for kw in processed:
        freq[kw] = freq.get(kw, 0) + 1
    return freq

In [25]:
from bs4 import BeautifulSoup

def extract_available_tags(html_text):
    """
    Extracts and returns a list of unique HTML tag names present in the given html_text.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    tags = set([tag.name for tag in soup.find_all()])
    useless_tags = {'script', 'style', 'noscript', 'head', 'iframe'}
    tags = tags - useless_tags
    return sorted(tags)

In [26]:
# 5. Aggregate and Print URL Ratings Across All Pages (as a pseudo-tag for sorting)
from urllib.parse import urlparse, unquote
from collections import defaultdict
import urllib.request

tag_rating_sum = defaultdict(float)
tag_count = defaultdict(int)

for i in range(num_pages):
    gt_url = f"{base_url}/{i}/GT.txt"
    html_url = f"{base_url}/{i}/HTML.txt"
    url_file_url = f"{base_url}/{i}/URL.txt"
    processed_keywords = []
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Copilot/1.0)"}
    gt_req = urllib.request.Request(gt_url, headers=headers)
    html_req = urllib.request.Request(html_url, headers=headers)
    url_req = urllib.request.Request(url_file_url, headers=headers)
    try:
        with urllib.request.urlopen(gt_req, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            gt_keywords = gt_text.split()
            processed_keywords = list(set(process_keywords(gt_keywords)))
            # print(f"Page {i}: Processed GT keywords: {processed_keywords}")
    except Exception as e:
        print(f"Error processing GT for page {i}: {e}")
        continue
    # HTML tag ratings
    try:
        with urllib.request.urlopen(html_req, timeout=5) as html_response:
            html_text = html_response.read().decode("utf-8-sig").strip()
            tags = extract_available_tags(html_text)
            for tag in tags:
                result = extract_and_process_keywords_from_tag(html_text, tag)
                # print(f"{tag}: {result}")
                total_keywords = sum(result.values())
                matched_sum = sum(freq for kw, freq in result.items() if kw in processed_keywords)
                rating = matched_sum / total_keywords if total_keywords > 0 else 0
                tag_rating_sum[tag] += rating
                tag_count[tag] += 1
    except Exception as e:
        print(f"Error processing HTML for page {i}: {e}")
        continue
    # URL rating as a pseudo-tag
    try:
        with urllib.request.urlopen(url_req, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()
            parsed_url = urlparse(real_url)
            normalized_path = unquote(parsed_url.path.lower())
            url_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)
            processed_url_keywords = process_keywords(url_tokens)
            total_url_keywords = len(processed_url_keywords)
            matched_url_keywords = sum(1 for kw in processed_url_keywords if kw in processed_keywords)
            rating = matched_url_keywords / total_url_keywords if total_url_keywords > 0 else 0
            tag_rating_sum['URL'] += rating
            tag_count['URL'] += 1
    except Exception as e:
        print(f"Error processing URL for page {i}: {e}")
        continue

# Calculate sum rating per tag and sort (including URL as a tag)
tag_sum_rating = {tag: tag_rating_sum[tag] for tag in tag_rating_sum}
sorted_tags = sorted(tag_sum_rating.items(), key=lambda x: x[1], reverse=True)

print("Tag and URL rating sum across all pages (sorted):")
for tag, rating_sum in sorted_tags:
    print(f"{tag}: {rating_sum:.4f}")

Tag and URL rating sum across all pages (sorted):
title: 41.8507
h1: 26.3131
URL: 18.9299
h2: 12.9877
a: 9.8007
html: 9.7465
body: 9.2142
li: 8.6534
div: 8.6341
ul: 8.4722
p: 7.7600
span: 7.6541
nav: 6.4883
header: 6.4320
h3: 5.7492
h4: 4.4459
ol: 3.9105
main: 3.6141
strong: 3.6019
b: 3.0750
footer: 3.0470
section: 3.0068
form: 2.6056
button: 2.3340
article: 1.9319
option: 1.7775
select: 1.7775
label: 1.7715
em: 1.6999
table: 1.5918
tr: 1.5388
aside: 1.4122
td: 1.3254
figure: 1.0253
figcaption: 1.0253
svg: 1.0000
dd: 0.9894
fieldset: 0.9557
address: 0.8889
small: 0.8207
dl: 0.7095
th: 0.6806
caption: 0.6667
font: 0.6647
center: 0.6463
thead: 0.5416
legend: 0.5333
optgroup: 0.4862
h5: 0.4857
tbody: 0.3253
h6: 0.2857
abbr: 0.2500
i: 0.2314
html-teaser: 0.1579
dt: 0.1500
blockquote: 0.0556
drawer-navigation: 0.0500
app-root: 0.0169
br: 0.0000
circle: 0.0000
img: 0.0000
input: 0.0000
line: 0.0000
link: 0.0000
meta: 0.0000
path: 0.0000
picture: 0.0000
source: 0.0000
base: 0.0000
use: 0.0000